In [ ]:
from pathlib import Path
import modal
project_name = 'language-model-pretraining'
LOCAL_STORAGE_ROOT = Path('/Users/oguz/Projects/transformer/volume')
app = modal.App(project_name)

container = (
    modal.Image.debian_slim(python_version="3.12") 
    .uv_sync(uv_project_dir="/Users/oguz/Projects/transformer")
    .add_local_python_source("transformer")
)

volume = modal.Volume.from_name('language-model-pretraining', create_if_missing=True)
@app.cls(image=container, volumes={'/storage' : volume})
class Pipeline:
    @modal.enter()
    def setup(self):
        self.storage_root = Path('/storage') if not modal.is_local() else LOCAL_STORAGE_ROOT
        print (self.storage_root)
    @modal.method()
    def show(self):
        for f in (self.storage_root / 'data').iterdir():
            print (f)


# with modal.enable_output():
with app.run():
    Pipeline().show.remote()
with app.run():
    Pipeline().show.local()


In [ ]:
from pathlib import Path
from config import LOCAL_VOLUME, PROJECT_ROOT

import modal



project_name = 'LLM-pretraining'
app = modal.App()

container = (
    modal.Image.debian_slim(python_version="3.12") 
    .uv_sync(uv_project_dir=PROJECT_ROOT)
    .add_local_python_source("transformer")
)

volume = modal.Volume.from_name('LLM-pretraining', create_if_missing=True,)


## PREPARE THE DATA 
dataset = 'gutenberg'
data_sources = {
'train' : [
    "https://gutenberg.org/cache/epub/1184/pg1184.txt",
],
'valid' : [
    "https://gutenberg.org/cache/epub/1513/pg1513.txt",
]}
special_tokens = ["<|endoftext|>", "<|begin|>", "<|end|>"]
token_source = "data/{dataset}/train.txt"
vocabulary_size = 1000


@app.cls(image=container, volumes={'/storage' : volume})
class Pipeline:
    @modal.enter()
    def setup(self):
        from transformer.util import download_and_concat, prepare_tokenizer
        from transformer import Tokenizer
        self.VOLUME = Path('/storage') if not modal.is_local() else LOCAL_VOLUME
        # download text to volume.
        for type, urls in data_sources.items():
            download_and_concat(urls, f"data/{dataset}/{type}.txt", self.VOLUME, separator="\n<|endoftext|>\n")
        prepare_tokenizer('tokenizer_test', vocabulary_size, special_tokens, token_source, self.VOLUME)
        tokenizer =  Tokenizer.from_files('tokenizer_test', self.VOLUME)


    @modal.method()
    def show(self):
        for f in (self.storage_root / 'data').iterdir():
            print (f)

PosixPath('/Users/oguz/Projects/transformer')